# Refine

> Postprocess markdown files by fixing heading hierarchy and describint images

In [ ]:
#| default_exp refine

This module aims to fix and enrich markdown headings from OCR'd PDF files by:

1. Fixing heading hierarchy that was corrupted during OCR
2. Optionnaly, adding page numbers to headings for better navigation
3. Describing figures

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs
from re import sub, findall, MULTILINE
from pydantic import BaseModel
from lisette import *
from lisette.core import completion
from typing import Callable
import os
import json
import shutil
from asyncio import Semaphore, gather, sleep

## Heading Hierarchy

Functions for detecting and fixing markdown heading levels



OCR'd PDF files often have corrupted heading hierarchies - headings may jump levels incorrectly (e.g., from H1 to H4) or use inconsistent levels for sections at the same depth. This section provides tools to automatically detect and fix these issues using LLMs, while also optionally adding page numbers for easier navigation.

The first step is extracting all headings from a markdown document so we can analyze their structure.

In [ ]:
#| export
def get_hdgs(
    md:str # Markdown file string
    ) -> L: # L of strings
    "Return the markdown headings"
    # Sanitize removing '#' in python snippet if any
    md = sub(r'```[\s\S]*?```', '', md)
    return L(findall(r'^#{1,6} .+$', md, MULTILINE))



In [ ]:
#| export
def add_pg_hdgs(
    md:str, # Markdown file string, 
    n:int # Page number
    ) -> str: # Markdown file string
    "Add page number to all headings in page markdown"
    md = sub(r'```[\s\S]*?```', '', md)
    def repl(m): return m.group(0) + f' ... page {n}'
    return sub(r'^#{1,6} .+$', repl, md, flags=MULTILINE)

The `add_pg_hdgs` function serves two important purposes:

**1. Creating unique heading identifiers**

When fixing heading hierarchies across an entire document, we need a way to distinguish between headings that have the same text but appear in different locations. For example, a document might have multiple "Introduction" or "Conclusion" headings in different chapters. By appending the page number to each heading, we create unique identifiers that allow us to build a lookup table mapping each specific heading instance to its corrected version. This assumes (reasonably) that the same heading text won't appear twice on a single page.

**2. Providing spatial context for LLMs**

Adding page numbers gives LLMs valuable positional information when analyzing the document structure. The page number helps the model understand:
- Where a heading sits in the overall document flow
- The relative distance between sections
- Whether headings that seem related are actually close together or far apart

This spatial awareness can significantly improve the LLM's ability to infer the correct hierarchical relationships between headings, especially in long documents where similar section names might appear at different structural levels.

For instance:

In [ ]:
#| eval: false
pgs = read_pgs('files/test/md_all/resnet', join=False)
pg0,pg0_with = pgs[0][:500],add_pg_hdgs(pgs[0], n=1)[:500]
print('Before:\n' + 80*'-' + f'\n{pg0}\n\nAfter:\n' + 80*'-' + f'\n{pg0_with}')

Before:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition 

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, instead of learning unr

After:
--------------------------------------------------------------------------------
# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


#### Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the train

In [ ]:
#| export
def read_pgs_pg(
    path:str # Path to the markdown file
    ) -> L: # List of markdown pages
    "Read all pages of a markdown file and add page numbers to all headings"
    pgs = read_pgs(path, join=False)
    return L([add_pg_hdgs(pg, n) for n, pg in enumerate(pgs, 1)]).concat()

In [ ]:
#| eval: false
pgs = read_pgs_pg('files/test/md_all/resnet')
hdgs = L([get_hdgs(pg) for pg in pgs]).concat()
hdgs

(#22) ['# Deep Residual Learning for Image Recognition  ... page 1','#### Abstract ... page 1','## 1. Introduction ... page 1','## 2. Related Work ... page 2','## 3. Deep Residual Learning ... page 3','### 3.1. Residual Learning ... page 3','### 3.2. Identity Mapping by Shortcuts ... page 3','### 3.3. Network Architectures ... page 3','### 3.4. Implementation ... page 4','## 4. Experiments ... page 4','### 4.1. ImageNet Classification ... page 4','### 4.2. CIFAR-10 and Analysis ... page 7','### 4.3. Object Detection on PASCAL and MS COCO ... page 8','## References ... page 9','## A. Object Detection Baselines ... page 10','## PASCAL VOC ... page 10','## MS COCO ... page 10','## B. Object Detection Improvements ... page 10','## MS COCO ... page 10','## PASCAL VOC ... page 11'...]

To make it easier for an LLM to reference specific headings when suggesting fixes, we format them with index numbers.

In [ ]:
#| export
def fmt_hdgs_idx(
    hdgs: list[str] # List of markdown headings
    ) -> str: # Formatted string with index
    "Format the headings with index"
    return '\n'.join(f"{i}. {h}" for i, h in enumerate(hdgs))


In [ ]:
#| eval: false
hdgs_fmt = fmt_hdgs_idx(hdgs)
print(hdgs_fmt)

0. # Deep Residual Learning for Image Recognition  ... page 1
1. #### Abstract ... page 1
2. ## 1. Introduction ... page 1
3. ## 2. Related Work ... page 2
4. ## 3. Deep Residual Learning ... page 3
5. ### 3.1. Residual Learning ... page 3
6. ### 3.2. Identity Mapping by Shortcuts ... page 3
7. ### 3.3. Network Architectures ... page 3
8. ### 3.4. Implementation ... page 4
9. ## 4. Experiments ... page 4
10. ### 4.1. ImageNet Classification ... page 4
11. ### 4.2. CIFAR-10 and Analysis ... page 7
12. ### 4.3. Object Detection on PASCAL and MS COCO ... page 8
13. ## References ... page 9
14. ## A. Object Detection Baselines ... page 10
15. ## PASCAL VOC ... page 10
16. ## MS COCO ... page 10
17. ## B. Object Detection Improvements ... page 10
18. ## MS COCO ... page 10
19. ## PASCAL VOC ... page 11
20. ## ImageNet Detection ... page 11
21. ## C. ImageNet Localization ... page 12


We use a Pydantic model to ensure the LLM returns corrections in a structured format - a dictionary mapping heading indices to their corrected versions.

In [ ]:
#| export
class HeadingCorrections(BaseModel):
    corrections: dict[int, str]  # index → corrected heading

This prompt instructs the LLM on what types of heading hierarchy errors to fix while preserving the document's intended structure. It focuses on three main issues: 

- level jumps that skip intermediate levels, 
- numbering inconsistencies where subsection depth doesn't match heading level, and
- ensures decreasing levels (moving back up the hierarchy) are preserved.

In [ ]:
#| export
prompt_fix_hdgs = """Fix markdown heading hierarchy errors while preserving the document's intended structure.

INPUT FORMAT: Each heading is prefixed with its index number (e.g., "0. # Title ... page 1")

RULES - Apply these fixes in order:

1. **Single H1 rule**: Documents must have exactly ONE # heading (typically the document title at the top)
   - If index 0 is already #, then all subsequent headings (index 1+) must be ## or deeper
   - If no H1 exists, the first major heading should be #, and all others ## or deeper
   - NO exceptions: appendices, references, and all sections are ## or deeper after the title

2. **Infer depth from numbering patterns**: If headings contain section numbers, deeper nesting means deeper heading level
   - Parent section (e.g., "1", "2", "A") should be shallower than child (e.g., "1.1", "2.a", "A.1")
   - Child section should be one # deeper than parent
   - Works with any numbering: "1/1.1/1.1.1", "A/A.1/A.1.a", "I/I.A/I.A.1", etc.

3. **Level jumps**: Headings can only increase by one # at a time when moving deeper
   - Wrong: ## Section → ##### Subsection
   - Fixed: ## Section → ### Subsection

4. **Decreasing levels is OK**: Moving back up the hierarchy (### to ##) is valid for new sections

OUTPUT: Return a Python dictionary mapping index to corrected heading (without the index prefix).
IMPORTANT: Preserve the " ... page N" suffix in all corrected headings.
Only include entries that need changes.

Headings to analyze:
{headings_list}
"""


#| export
Now we can use an LLM to automatically detect and fix these heading hierarchy issues. The function uses litellm (wrapped by the [Lisette package](https://lisette.answer.ai) to send the formatted headings to a language model along with our correction rules. The LLM analyzes the structure and returns only the headings that need fixing, mapped by their index numbers.

In [ ]:
#| export
def fix_hdg_hierarchy(
    hdgs: list[str], # List of markdown headings
    prompt: str=None, # Prompt to use
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None # API key
    ) -> dict[int, str]: # Dictionary of index → corrected heading
    "Fix the heading hierarchy"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    if prompt is None: prompt = prompt_fix_hdgs
    prompt = prompt.format(headings_list=fmt_hdgs_idx(hdgs))
    r = completion(model=model, messages=[{"role": "user", "content": prompt}], response_format=HeadingCorrections, api_key=api_key)
    return json.loads(r.choices[0].message.content)['corrections']


In [ ]:
#| eval: false
fixes = fix_hdg_hierarchy(hdgs)
fixes

{'1': '## Abstract ... page 1',
 '13': '## References ... page 9',
 '14': '## Appendix A. Object Detection Baselines ... page 10',
 '15': '### PASCAL VOC ... page 10',
 '16': '### MS COCO ... page 10',
 '17': '## Appendix B. Object Detection Improvements ... page 10',
 '18': '### MS COCO ... page 10',
 '19': '### PASCAL VOC ... page 11',
 '20': '### ImageNet Detection ... page 11',
 '21': '## Appendix C. ImageNet Localization ... page 12'}

The corrections come back as string indices, but we need to map the actual heading text to its corrected version for easy replacement in the document.

In [ ]:
#| export
@delegates(fix_hdg_hierarchy)
def mk_fixes_lut(
    hdgs: list[str], # List of markdown headings
    model: str='claude-sonnet-4-5', # Model to use
    api_key: str=None, # API key
    **kwargs
    ) -> dict[str, str]: # Dictionary of old → new heading
    "Make a lookup table of fixes"
    if api_key is None: api_key = os.getenv('ANTHROPIC_API_KEY')
    fixes = fix_hdg_hierarchy(hdgs, model=model, api_key=api_key, **kwargs)
    return {hdgs[int(k)]:v for k,v in fixes.items()}

In [ ]:
#| eval: false
lut_fixes = mk_fixes_lut(hdgs)
lut_fixes

{'#### Abstract ... page 1': '## Abstract ... page 1',
 '## References ... page 9': '## References ... page 9',
 '## A. Object Detection Baselines ... page 10': '## Appendix A. Object Detection Baselines ... page 10',
 '## PASCAL VOC ... page 10': '### PASCAL VOC ... page 10',
 '## MS COCO ... page 10': '### MS COCO ... page 10',
 '## B. Object Detection Improvements ... page 10': '## Appendix B. Object Detection Improvements ... page 10',
 '## PASCAL VOC ... page 11': '### PASCAL VOC ... page 11',
 '## ImageNet Detection ... page 11': '### ImageNet Detection ... page 11',
 '## C. ImageNet Localization ... page 12': '## Appendix C. ImageNet Localization ... page 12'}

Now we can apply the fixes to individual pages. We optionally add page numbers to headings for easier navigation in the final document.

In [ ]:
#| export
def apply_hdg_fixes(
    p:str, # Page to fix
    lut_fixes: dict[str, str], # Lookup table of fixes
    ) -> str: # Page with fixes applied
    "Apply the fixes to the page"
    for old in get_hdgs(p): p = p.replace(old, lut_fixes.get(old, old))
    return p

In [ ]:
#| eval: false
pg_nb = 1
p = read_pgs_pg('files/test/md_all/resnet')[0]
print(apply_hdg_fixes(p, lut_fixes)[:300])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framewor


Finally, we tie everything together in a single function that processes an entire document directory, fixing all heading hierarchy issues and optionally adding page numbers.

In [ ]:
fix_hdgs('md', dst='md_fixed')

In [ ]:
#| export
@delegates(mk_fixes_lut)
def fix_hdgs(src:str, model:str='claude-sonnet-4-5', dst:str=None, img_folder:str='img', **kwargs):
    "Fix heading hierarchy in markdown document"
    src_path,dst_path = Path(src),Path(dst) if dst else Path(src)
    if dst_path != src_path: dst_path.mkdir(parents=True, exist_ok=True)
    src_imgs = src_path/img_folder
    if src_imgs.exists() and dst_path != src_path: shutil.copytree(src_imgs, dst_path/img_folder, dirs_exist_ok=True)
    pgs_with_pg = read_pgs_pg(src_path)
    lut = mk_fixes_lut(L([get_hdgs(pg) for pg in pgs_with_pg]).concat(), model, **kwargs)
    for i,p in enumerate(pgs_with_pg, 1): (dst_path/f'page_{i}.md').write_text(apply_hdg_fixes(p, lut))

In [ ]:
#| eval: false
fix_hdgs('files/test/md_all/resnet', dst='files/test/md_fixed/resnet')

In [ ]:
#| eval: false
!ls -R 'files/test/md_fixed/resnet'

files/test/md_fixed/resnet:
img	   page_10.md  page_12.md  page_3.md  page_5.md  page_7.md  page_9.md
page_1.md  page_11.md  page_2.md   page_4.md  page_6.md  page_8.md

files/test/md_fixed/resnet/img:
img-0.jpeg  img-2.jpeg	img-4.jpeg  img-6.jpeg
img-1.jpeg  img-3.jpeg	img-5.jpeg


In [ ]:
#| eval: false
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


## Image Description

Tools for classifying and describing images in markdown documents

In [ ]:
#| eval: false
# Dev only
import sys
sys.path.append('../../..')
from ctx_utils import *

In [ ]:
mappr = nb_to_md('_06_mappr.ipynb')
mappr[:200]

'# Mappr\n\n> Scale up evaluation report mapping against evaluation frameworks using agentic workflows\n\n\n---\n\n::: {.callout-warning}\nThis notebook is a work in progress.\n:::\n\n---\n\nManually mapping evalua'

In [ ]:
md = read_pgs('files/test/md_fixed/resnet')
print(md[:500])

# Deep Residual Learning for Image Recognition  ... page 1

Kaiming He Xiangyu Zhang Shaoqing Ren Jian Sun<br>Microsoft Research<br>\{kahe, v-xiangz, v-shren, jiansun\}@microsoft.com


## Abstract ... page 1

Deeper neural networks are more difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions with reference to the layer inputs, ins


# Lisette


<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

> **NB**: If you are reading this in GitHub’s readme, we recommend you
> instead read the much more nicely formatted [documentation
> format](https://lisette.answer.ai/) of this tutorial.

*Lisette* is a wrapper for the [LiteLLM Python
SDK](https://docs.litellm.ai/), which provides unified access to 100+
LLM providers using the OpenAI API format.

LiteLLM provides a unified interface to access multiple LLMs, but it’s
quite low level: it leaves the developer to do a lot of stuff manually.
Lisette automates pretty much everything that can be automated, whilst
providing full control. Amongst the features provided:

- A [`Chat`](https://lisette.answer.ai/core.html#chat) class that
  creates stateful dialogs across any LiteLLM-supported model
- Convenient message creation utilities for text, images, and mixed
  content
- Simple and convenient support for tool calling with automatic
  execution
- Built-in support for web search capabilities (including citations for
  supporting models)
- Streaming responses with formatting
- Full async support with
  [`AsyncChat`](https://lisette.answer.ai/core.html#asyncchat)
- Prompt caching (for supporting models)

To use Lisette, you’ll need to set the appropriate API keys as
environment variables for whichever LLM providers you want to use.

## Get started

LiteLLM will automatically be installed with Lisette, if you don’t
already have it.

``` python
!pip install lisette -qq
```

Lisette only exports the symbols that are needed to use the library, so
you can use import \* to import them. Here’s a quick example showing how
easy it is to switch between different LLM providers:

``` python
from lisette import *
```

## Chat

``` python
models = ['claude-sonnet-4-20250514', 'gemini/gemini-2.5-flash', 'openai/gpt-4o']

for model in models:
    chat = Chat(model)
    res = chat("Please tell me about yourself in one brief sentence.")
    display(res)
```

I’m Claude, an AI assistant created by Anthropic to be helpful,
harmless, and honest in conversations and tasks.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=29, prompt_tokens=17, total_tokens=46, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

I am a large language model, trained by Google, designed to assist with
information and generate text.

<details>

- id: `chatcmpl-xxx`
- model: `gemini-2.5-flash`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=603, prompt_tokens=11, total_tokens=614, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=583, rejected_prediction_tokens=None, text_tokens=20), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=None, text_tokens=11, image_tokens=None))`

</details>

I’m an AI language model created by OpenAI, designed to assist with a
wide range of questions and tasks by providing information and
generating text-based responses.

<details>

- id: `chatcmpl-xxx`
- model: `gpt-4o-2024-08-06`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=30, prompt_tokens=17, total_tokens=47, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None))`

</details>

That’s it! Lisette handles all the provider-specific details
automatically. Each model will respond in its own style, but the
interface remains the same.

## Message formatting

### Multiple messages

Lisette accepts multiple messages in one go:

``` python
chat = Chat(models[0])
res = chat(['Hi! My favorite drink coffee.', 'Hello!', 'Whats my favorite drink?'])
display(res)
```

Hello! Based on what you just told me, your favorite drink is coffee! ☕

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=22, prompt_tokens=23, total_tokens=45, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

If you have a pre-existing message history, you can also pass it when
you create the [`Chat`](https://lisette.answer.ai/core.html#chat)
object:

``` python
chat = Chat(models[0],hist=['Hi! My favorite drink is coffee.', 'Hello!'])
res = chat('Whats my favorite drink?')
display(res)
```

Your favorite drink is coffee! You just mentioned that in your previous
message.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=18, prompt_tokens=30, total_tokens=48, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

### Images

Lisette also makes it easy to include images in your prompts:

``` python
from pathlib import Path
from IPython.display import Image
```

``` python
fn = Path('samples/puppy.jpg')
img = fn.read_bytes()
Image(img)
```

![](index_files/figure-commonmark/cell-8-output-1.jpeg)

All you have to do is read it in as bytes:

``` python
img[:20]
```

    b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00'

And you can pass it inside a
[`Chat`](https://lisette.answer.ai/core.html#chat) object:

``` python
chat = Chat(models[0])
chat([img, "What's in this image? Be brief."])
```

A cute puppy with brown and white fur lying on grass next to purple
flowers.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=20, prompt_tokens=108, total_tokens=128, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

### Prefill

Some providers (e.g. Anthropic) support `prefill`, allowing you to
specify how the assistant’s response should begin:”

``` python
chat = Chat(models[0])
chat("Concisely, what's the meaning of life?", prefill="According to Douglas Adams,")
```

According to Douglas Adams,it’s 42.

More seriously, there’s no universal answer. Common perspectives
include: - Creating meaning through relationships, growth, and
contribution - Fulfilling a divine purpose or spiritual calling -
Maximizing well-being and minimizing suffering - Leaving a positive
legacy - Simply experiencing and appreciating existence itself

The meaning might be something you create rather than discover.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=84, prompt_tokens=24, total_tokens=108, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

## Tools

Lisette makes it easy to give LLMs access to Python functions. Just
define a function with type hints and a docstring:

``` python
def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b
```

Now pass the function to
[`Chat`](https://lisette.answer.ai/core.html#chat) and the model can use
it automatically:

``` python
chat = Chat(models[0], tools=[add_numbers])
res = chat("What's 47 + 23? Use the tool.")
res
```

The result of 47 + 23 is 70.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=17, prompt_tokens=533, total_tokens=550, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

If you want to see all intermediate messages and outputs you can use the
`return_all=True` feature.

``` python
chat = Chat(models[0], tools=[add_numbers])
res = chat("What's 47 + 23 + 59? Use the tool.",max_steps=3,return_all=True)
display(*res)
```

I’ll help you calculate 47 + 23 + 59 using the add_numbers tool. Since
the tool can only add two numbers at a time, I’ll need to do this in two
steps.

🔧 add_numbers({“a”: 47, “b”: 23})

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `tool_calls`
- usage:
  `Usage(completion_tokens=116, prompt_tokens=433, total_tokens=549, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

    {'tool_call_id': 'toolu_01F9oakoP8ANHkTMD1DyQDi7',
     'role': 'tool',
     'name': 'add_numbers',
     'content': '70'}

Now I’ll add the result (70) to the third number (59):

🔧 add_numbers({“a”: 70, “b”: 59})

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `tool_calls`
- usage:
  `Usage(completion_tokens=87, prompt_tokens=562, total_tokens=649, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

    {'tool_call_id': 'toolu_01Cdf3FHJdbx64F8H8ooE1Db',
     'role': 'tool',
     'name': 'add_numbers',
     'content': '129'}

The answer is 129. So 47 + 23 + 59 = 129.

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=25, prompt_tokens=662, total_tokens=687, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

It shows the intermediate tool calls, and the tool results!

## Web search

Some models support web search capabilities. Lisette makes this easy to
use:

``` python
chat = Chat(models[0], search='l')  # 'l'ow, 'm'edium, or 'h'igh search context
res = chat("Please tell me one fun fact about otters. Keep it brief")
res
```

Here’s a fun fact about otters: Sea otters allow themselves to get
entangled in kelp forests - this creates a tether so they don’t drift
away on sleep currents as they sleep. They essentially use kelp as a
natural anchor to stay in place while floating and resting on the
water’s surface!

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=143, prompt_tokens=15626, total_tokens=15769, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), server_tool_use=ServerToolUse(web_search_requests=1), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

> [!TIP]
>
> Some providers (like Anthropic) provide citations for their search
> results.

``` python
res.choices[0].message.provider_specific_fields
```

    {'citations': [[{'type': 'web_search_result_location',
        'cited_text': 'Sea Otters allow themselves to get entangled in kelp forests this creates a tether so they don’t drift away on sleep currents as they sleep. ',
        'url': 'https://www.mygreenworld.org/blog/facts-about-otters',
        'title': 'Five Fast Facts about Otters — My Green World',
        'encrypted_index': 'EpABCioIBxgCIiQ4ODk4YTFkYy0yMTNkLTRhNmYtOTljYi03ZTBlNTUzZDc0NWISDCMi/kxdYrQXVUX+ZxoMVvW3BHE29cyMhwAFIjBZEBw3PaH+XAslsXWMNucD7FqSwe5Fnnsfh2RzTX9x/q9XQ1Mm1Ke6JOreehNzVI0qFDkJYT4NCX8U4CjHHwoyLKtY66vhGAQ='}]],
     'thinking_blocks': None}

## Streaming

For real-time responses, use `stream=True` to get chunks as they’re
generated rather than waiting for the complete response:

``` python
chat = Chat(models[0])
res_gen = chat("Concisely, what are the top 10 biggest animals?", stream=True)
res_gen
```

    <generator object Chat._call>

``` python
from litellm import ModelResponse, ModelResponseStream
```

You can loop over the generator to get the partial responses:

``` python
for chunk in res_gen:
    if isinstance(chunk,ModelResponseStream): print(chunk.choices[0].delta.content,end='')
```

    Here are the top 10 biggest animals by size/weight:

    1. **Blue whale** - largest animal ever, up to 100 feet long
    2. **Fin whale** - second-largest whale, up to 85 feet
    3. **Bowhead whale** - up to 65 feet, very heavy build
    4. **Right whale** - up to 60 feet, extremely bulky
    5. **Sperm whale** - up to 67 feet, largest toothed whale
    6. **Gray whale** - up to 50 feet
    7. **Humpback whale** - up to 52 feet
    8. **African elephant** - largest land animal, up to 13 feet tall
    9. **Colossal squid** - up to 46 feet long (largest invertebrate)
    10. **Giraffe** - tallest animal, up to 18 feet tall

    *Note: Various whale species dominate due to the ocean's ability to support massive body sizes.*None

And the final chunk is the complete `ModelResponse`:

``` python
chunk
```

Here are the top 10 biggest animals by size/weight:

1.  **Blue whale** - largest animal ever, up to 100 feet long
2.  **Fin whale** - second-largest whale, up to 85 feet
3.  **Bowhead whale** - up to 65 feet, very heavy build
4.  **Right whale** - up to 60 feet, extremely bulky
5.  **Sperm whale** - up to 67 feet, largest toothed whale
6.  **Gray whale** - up to 50 feet
7.  **Humpback whale** - up to 52 feet
8.  **African elephant** - largest land animal, up to 13 feet tall
9.  **Colossal squid** - up to 46 feet long (largest invertebrate)
10. **Giraffe** - tallest animal, up to 18 feet tall

*Note: Various whale species dominate due to the ocean’s ability to
support massive body sizes.*

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=233, prompt_tokens=22, total_tokens=255, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None)`

</details>

## Async

For web applications and concurrent operations, like in
[FastHTML](https://fastht.ml), we recommend using
[`AsyncChat`](https://lisette.answer.ai/core.html#asyncchat):

``` python
chat = AsyncChat(models[0])
await chat("Hi there")
```

Hello! How are you doing today? Is there anything I can help you with?

<details>

- id: `chatcmpl-xxx`
- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage:
  `Usage(completion_tokens=20, prompt_tokens=9, total_tokens=29, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

To wrap up, we’ll show an example of async + streaming + toolcalling +
search:

``` python
chat = AsyncChat(models[0], search='l', tools=[add_numbers])
res = await chat("""\
Search the web for the avg weight, in kgs, of male African and Asian elephants. Then add the two.
Keep your replies ultra concise! Dont search the web more than once please.
""", max_steps=4, stream=True)
await adisplay_stream(res)  # this is a convenience function to make async streaming look great in notebooks!
```

Based on the search results:

**Male African elephants**:
[\*](https://www.africa-safaris.com/How-Much-Does-An-Elephant-Weigh "How Much Does An Elephant Weigh")
[\*](https://www.quora.com/What-is-the-average-weight-of-an-adult-African-elephant-in-pounds-and-tons "What is the average weight of an adult African elephant in pounds and tons? - Quora")
Average weight is 5,000 kg (11,000 pounds)

**Male Asian elephants**:
[\*](https://www.ifaw.org/international/journal/difference-african-asian-elephants "African Elephants vs. Asian Elephants | IFAW")
[\*](https://www.ifaw.org/international/journal/difference-african-asian-elephants "African Elephants vs. Asian Elephants | IFAW")
Average weight is 3,600 kg (7,900 pounds)
<details class="tool-usage-details">

`add_numbers({"a": 5000, "b": 3600})` - `8600`

</details>

**Total**: 8,600 kg

## Next steps

Ready to dive deeper?

- Check out the rest of the
  [documentation](https://lisette.answer.ai/core.html).
- Visit the [GitHub repository](https://github.com/answerdotai/lisette)
  to contribute or report issues.
- Join our [Discord community](https://discord.gg/y7cDEX7r)!

My task here is to describe the images downloaded by the OCR and injecting those descriptions in the markdown file. Please refer to $`mappr` package to see how to leverage lisette/litellm as well as to the lisette doc

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's start by finding image references in the markdown. We'll need to extract the image paths and their locations:

```python
def find_imgs(md:str) -> L:
    "Find all image references in markdown"
    return L(findall(r'!\[([^\]]*)\]\(([^)]+)\)', md))
```

Let's test it on the resnet markdown:

```python
md = read_pgs('files/test/md_fixed/resnet')
imgs = find_imgs(md)
imgs[:3]
```

In [ ]:
def find_imgs(md:str) -> L:
    "Find all image references in markdown"
    return L(findall(r'!\[([^\]]*)\]\(([^)]+)\)', md))

In [ ]:
imgs = Path('files/test/md_fixed/resnet/img').ls(file_exts='.jpeg')
imgs

(#7) [Path('files/test/md_fixed/resnet/img/img-5.jpeg'),Path('files/test/md_fixed/resnet/img/img-0.jpeg'),Path('files/test/md_fixed/resnet/img/img-1.jpeg'),Path('files/test/md_fixed/resnet/img/img-2.jpeg'),Path('files/test/md_fixed/resnet/img/img-3.jpeg'),Path('files/test/md_fixed/resnet/img/img-6.jpeg'),Path('files/test/md_fixed/resnet/img/img-4.jpeg')]

In [ ]:
md = read_pgs('files/test/md_fixed/resnet')
imgs_str = find_imgs(md)
imgs_str

(#7) [('img-0.jpeg', 'img-0.jpeg'),('img-1.jpeg', 'img-1.jpeg'),('img-2.jpeg', 'img-2.jpeg'),('img-3.jpeg', 'img-3.jpeg'),('img-4.jpeg', 'img-4.jpeg'),('img-5.jpeg', 'img-5.jpeg'),('img-6.jpeg', 'img-6.jpeg')]

In [ ]:
#| export
class ImgDescription(BaseModel):
    is_informative: bool
    description: str

In [ ]:
#| export
describe_img_prompt = """Analyze this image from an academic/technical document.

Step 1: Determine if this image is informative for understanding the document content.
- Informative: charts, diagrams, tables, technical illustrations, experimental results, architectural diagrams
- Non-informative: logos, decorative images, generic photos, page backgrounds

Step 2: 
- If informative: Provide a detailed description including the type of visualization, key elements and their relationships, important data or patterns, and relevant technical details.
- If non-informative: Provide a brief label (e.g., "Company logo", "Decorative header image")

Return your response as JSON with 'is_informative' (boolean) and 'description' (string) fields."""

In [ ]:
#| export
def parse_resp(result):
    "Extract JSON from Lisette response"
    return json.loads(result.choices[0].message.content)

In [ ]:
#| export
async def describe_img(
    img_path: Path,  # Path to the image file
    model: str = 'claude-sonnet-4-5',  # Model to use
    prompt: str = describe_img_prompt  # Prompt for description
) -> ImgDescription:
    "Describe a single image using AsyncChat"
    chat = AsyncChat(model=model)
    r = await chat([img_path.read_bytes(), prompt], response_format=ImgDescription)
    return r

In [ ]:
img_path = Path('files/test/md_fixed/resnet/img/img-0.jpeg')
r = await describe_img(img_path)
r

{"is_informative": true, "description": "This figure contains two side-by-side line graphs comparing the performance of 56-layer and 20-layer neural networks during training. \n\nLeft panel: Shows \"training error (%)\" on the y-axis (ranging from 0 to 20%) versus \"iter. (1e4)\" on the x-axis (ranging from 0 to 6). The 56-layer network (red line) maintains higher training error around 18-20% initially, then drops sharply around iteration 3e4 to stabilize around 6-8%. The 20-layer network (yellow/olive line) starts around 20% and decreases more gradually and consistently to reach approximately 2-3% by iteration 6e4.\n\nRight panel: Shows \"test error (%)\" on the y-axis (same scale) versus \"iter. (1e4)\" on the x-axis (same range). Both networks show similar patterns to their training curves but with the test errors remaining higher. The 56-layer network (red) stabilizes around 14-16% after its drop, while the 20-layer network (yellow/olive) decreases more smoothly to around 10-11%.\n\nThis visualization demonstrates a degradation problem where the deeper 56-layer network performs worse than the shallower 20-layer network, both in training and test error, which is counterintuitive to the expectation that deeper networks should perform better."}

<details>

- id: `chatcmpl-36370183-252b-4832-8e2c-911fbd48a4b5`
- model: `claude-sonnet-4-5-20250929`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=365, prompt_tokens=1074, total_tokens=1439, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`

</details>

In [ ]:
#| export
async def limit(semaphore, coro, delay=None):
    async with semaphore:
        r = await coro
        if delay: await sleep(delay)
        return r

In [ ]:
r = await limit(Semaphore(2), describe_img(img_path), delay=1)
r

NameError: name 'img_path' is not defined

Yes show me how it works with the "gather" function

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the pattern from your `mappr` code:

```python
# First call with delay
results = [await limit(semaphore, describe_img(imgs[0]), delay=1)]

# Remaining calls in parallel
remaining = await gather(*[limit(semaphore, describe_img(img), delay=1) for img in imgs[1:]])

# Combine
results.extend(remaining)
```

The first call runs alone, then the rest run in parallel (but still rate-limited by the semaphore).

Does this pattern make sense for our image batch processing?

Yeah, I remembered to have used this pattern in the mappr package. in particular because I was hitting maximum requests per minute from Claude. How to improve and streamline these so that we can simplify it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question! A few options to streamline:

**Option 1: Just use `gather` for everything**
```python
results = await gather(*[limit(semaphore, describe_img(img), delay) for img in imgs])
```
Simpler, but waits for all to complete.

**Option 2: Use `asyncio.as_completed`**
```python
tasks = [limit(semaphore, describe_img(img), delay) for img in imgs]
results = [await task for task in asyncio.as_completed(tasks)]
```
Get results as they complete (better for progress tracking).

The separate first call in `mappr` might have been for debugging or handling edge cases. Do you need that, or prefer one of these simpler approaches?

But what is the standard way and the best practices in doing such parallel call? Because to my understanding if you use for example option one or even option two we might eat the maximum request per minute because we'd launch potentially dozens of parallel requests together. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You're absolutely right! That's exactly why the semaphore is critical.

The **semaphore controls concurrency** - if you set `Semaphore(2)`, only 2 requests run at once, even if you have 100 images. The rest wait in queue.

The **delay** adds spacing between individual calls to avoid bursts.

So the best practice is:
```python
semaphore = Semaphore(2)  # Max 2 concurrent
results = await gather(*[limit(semaphore, describe_img(img), delay=1) for img in imgs])
```

This ensures:
- Only 2 requests active at once
- 1 second gap between each call starting
- Stays under rate limits (e.g., 50/min with delay=1.2s and semaphore=2)

Does this clarify how semaphore + delay work together to prevent hitting rate limits?

Yes, but what if I want to run 10 in parallel? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Then just increase the semaphore:

```python
semaphore = Semaphore(10)  # Max 10 concurrent
```

But check your API rate limits first! For example:
- Claude: 50 requests/min → with 10 parallel, you'd hit limit in ~30 seconds
- You might need `delay=6` (60s/10 = 6s spacing) to stay under 50/min

What rate limits does your API have?

In [ ]:
imgs[:2]

In [ ]:
sem, delay = Semaphore(2), 1
results = await gather(*[limit(sem, describe_img(img), delay) for img in imgs[:2]])

In [ ]:
imgs[0].name

'img-5.jpeg'

In [ ]:
parse_resp(results[0])

{'is_informative': True,
 'description': 'This figure contains three line graphs showing training error rates over iterations for different neural network architectures. \n\nLeft panel: Shows error rates for "plain" networks (plain-20, plain-32, plain-44, plain-56) plotted over approximately 6×10^4 iterations. The curves show different convergence behaviors, with the 56-layer and 20-layer networks labeled specifically. Error rates range from 0-20%.\n\nMiddle panel: Displays error rates for ResNet architectures (ResNet-20, ResNet-32, ResNet-44, ResNet-56, ResNet-110) over the same iteration range. Multiple colored lines show convergence patterns, with 110-layer and 20-layer networks specifically labeled. The curves generally show smoother convergence compared to plain networks.\n\nRight panel: Shows a comparison between two residual networks (residual-110 in purple and residual-1202 in black) over iterations. Both curves stabilize around 5-8% error, with residual-1202 showing slightly h

Ok I think it's time to wrap the parallel sync call in to a fastai-coding style function

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
async def describe_imgs(
    imgs: list[Path],
    model: str = 'claude-sonnet-4-5',
    prompt: str = describe_img_prompt,
    semaphore: int = 2,
    delay: float = 1
) -> dict[str, dict]:
    "Describe multiple images in parallel and return dict mapping filename to description"
    sem = Semaphore(semaphore)
    results = await gather(*[limit(sem, describe_img(img, model, prompt), delay) for img in imgs])
    return {img.name: parse_resp(r) for img, r in zip(imgs, results)}
```

In [ ]:
async def describe_imgs(
    imgs: list[Path],
    model: str = 'claude-sonnet-4-5',
    prompt: str = describe_img_prompt,
    semaphore: int = 2,
    delay: float = 1
) -> dict[str, dict]:
    "Describe multiple images in parallel and return dict mapping filename to description"
    sem = Semaphore(semaphore)
    results = await gather(*[limit(sem, describe_img(img, model, prompt), delay) for img in imgs])
    return {img.name: parse_resp(r) for img, r in zip(imgs, results)}

In [ ]:
rs = await describe_imgs(imgs[:2])

In [ ]:
rs

{'img-5.jpeg': {'is_informative': True,
  'description': 'This figure contains three line graphs showing training error rates over iterations for different neural network architectures. \n\nLeft panel: Shows error rates for "plain" networks (plain-20, plain-32, plain-44, plain-56) plotted over approximately 6×10^4 iterations. The curves show different convergence behaviors, with the 56-layer and 20-layer networks labeled specifically. Error rates range from 0-20%.\n\nMiddle panel: Displays error rates for ResNet architectures (ResNet-20, ResNet-32, ResNet-44, ResNet-56, ResNet-110) over the same iteration range. Multiple colored lines show convergence patterns, with 110-layer and 20-layer networks specifically labeled. The curves generally show smoother convergence compared to plain networks.\n\nRight panel: Shows a comparison between two residual networks (residual-110 in purple and residual-1202 in black) over iterations. Both curves stabilize around 5-8% error, with residual-1202 sh

It looks great! Do you think that we should write this JSON locally in the describe_imgs function or should we do it separately? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I'd do it separately - keeps functions focused on one thing. Something like:

```python
def save_img_descriptions(descriptions: dict, path: Path):
    "Save image descriptions to JSON file"
    path.write_text(json.dumps(descriptions, indent=2))
```

Then you can call it after `describe_imgs`. Want to add a function to load existing descriptions too (to skip already-processed images)?

In [ ]:
def save_img_descriptions(descriptions: dict, path: Path):
    "Save image descriptions to JSON file"
    path.write_text(json.dumps(descriptions, indent=2))

but let's think about the api. Users will:
1. ocr
2. then `fix_md_hdgs`
3. then describe images. How to make this last step a "single" step?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking! Following the `fix_md_hdgs` pattern, we'd want something like:

```python
async def describe_md_imgs(
    src: str,
    model: str = 'claude-sonnet-4-5',
    dst: str = None,
    img_folder: str = 'img',
    semaphore: int = 2,
    delay: float = 1
):
    "Describe images and inject descriptions into markdown"
    # Find images, describe new ones, update markdown pages
    pass
```

But we still need the function to inject descriptions into the markdown. Should we build that next, then tie it all together?

Yes it makes sense but let's step back a bit. What would be the best API for these three steps either individually or globally? Give me alternatives. 


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here are some API alternatives:

**Option 1: Separate independent functions (current)**
```python
ocr_pdf('doc.pdf', 'md_raw')
fix_md_hdgs('md_raw', 'md_fixed')
await describe_md_imgs('md_fixed', 'md_enriched')
```
Pros: Flexible, skip steps, inspect intermediate outputs
Cons: Manual path management

**Option 2: Single pipeline function**
```python
await process_pdf('doc.pdf', 'output', steps=['ocr','fix_hdgs','describe_imgs'])
```
Pros: Simple one-liner for common case
Cons: Less flexible, harder to debug

**Option 3: In-place operations (like fix_md_hdgs with dst=None)**
```python
ocr_pdf('doc.pdf', 'md')
fix_md_hdgs('md')  # updates in place
await describe_md_imgs('md')  # updates in place
```
Pros: Simpler paths, natural sequence
Cons: Can't compare before/after easily

**Option 4: Chained with intermediate dirs**
```python
await enrich_md('md_raw')  # does fix_hdgs + describe_imgs
```
Pros: Common post-OCR cleanup as single step
Cons: Less granular control

Which approach fits your workflow best?

What would be the fasctore, fastai way? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The fastcore way is **Option 3 with optional `dst`** - matching your `fix_md_hdgs` pattern:

```python
ocr_pdf('doc.pdf', 'md')
fix_md_hdgs('md', dst='md_fixed')  # or dst=None for in-place
await describe_md_imgs('md_fixed')  # or dst='md_enriched' to preserve original
```

Each function:
- Takes `src` 
- Optional `dst` (None = in-place)
- Uses `@delegates` for shared params
- Does one thing well

Users compose as needed. Simple, explicit, flexible.

Want to build `describe_md_imgs` following this pattern?

Yeah, it sounds good. Can we just question the choice of the names of those functions, again from the perspective of fastai/fastcore-coding style?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good point! Looking at fastcore style - shorter, action-focused names:

```python
ocr_pdf('doc.pdf', 'md')
fix_hdgs('md', dst='md_fixed')
await enrich_imgs('md_fixed')
```

Or even more concise:
```python
ocr('doc.pdf', 'md')
fix_hdgs('md')
await add_img_desc('md')
```

The fastai way favors brevity when context is clear. Which naming feels right to you?

Yes awesome. And what could be the name of a single function wrapping these three steps together? 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the fastai style, something simple like:

```python
await enrich_pdf('doc.pdf', 'output')
```

Or if you want to emphasize the markdown conversion:

```python
await pdf_to_md('doc.pdf', 'output', enrich=True)
```

The first one feels more fastai - short, action verb, clear intent. What do you think?

I like very much the second option. However, I'd like to question the ocr name. Do you think we should change it to ocr_pdf do emphasize that it's only applicable to PDF? Or the dog string will clarify it enought? 

In [ ]:
await pdf_to_md('doc.pdf', 'output', img_desc=True):
    ocr('doc.pdf', 'md')
    fix_hdgs('md')
    if img_desc: await add_img_desc('md')
    # read mardown
    #return markdown as a list of pages

SyntaxError: invalid syntax (947683841.py, line 1)

Fix and complete the function above in fastai style

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The following vars are undefined:

- mappr